#  Applied Task – Loan Approval Decision Agent

## Hybrid Rule-Based + LLM Explanation System

⚠️ Important
This is NOT a prompting exercise.
This is a system design and rule-engine exercise.

The LLM must NEVER decide the loan.
Your Python logic must decide first.

---

#  LEVEL 1 – Basic Hybrid Decision Engine

##  Objective

Build a simple Loan Approval Decision System that:

* Extracts loan amount from user input
* Matches it against predefined rules
* Selects one rule
* Uses LLM only to explain the decision

---

# 🏗 Required Architecture (Level 1)

User Input

↓

Extract amount

↓

Match rule condition

↓

Select first matching rule

↓

LLM explanation

↓

Structured final output


---

#  Step 1 – Create Loan Rules CSV (Level 1)

Your CSV must contain:

* section
* rule_description
* condition
* decision
* risk_level

### Example Rules

| section | rule_description         | condition       | decision      | risk_level |
| ------- | ------------------------ | --------------- | ------------- | ---------- |
| Loan    | Small loan auto approved | amount <= 5000  | APPROVED      | Low        |
| Loan    | Medium loan needs review | amount <= 20000 | MANUAL_REVIEW | Medium     |
| Loan    | High loan rejected       | amount > 20000  | REJECTED      | High       |

⚠️ Rules must support only:

* amount
* single comparison operator
* one numeric value

No AND.
No credit score.
Keep it simple.

---

#  Step 2 – Extract Loan Amount

Example input:

"I need a loan of 15000 dollars."

Your system must:

* Extract 15000
* Assign to variable: amount

Hint:
Use regex to extract first numeric value.

---

#  Step 3 – Implement Safe Condition Matching

You must support:

* <=
* <
* >
* > =

Example:

Condition: amount <= 5000

If amount = 4000 → True
If amount = 8000 → False

⚠️ Do NOT use eval().

Implement manual comparison logic.

---

#  Step 4 – Rule Selection Strategy (Level 1)

Loop through rules in order.

The first rule that returns True wins.

Stop checking after match.

---

#  Step 5 – LLM Explanation Layer

After selecting the rule:

Send to LLM:

* User input
* Selected rule
* Decision
* Risk level

Required Output Format:


Decision: <APPROVED / REJECTED / MANUAL_REVIEW>

Risk Level: <Low / Medium / High>

Reasoning: <Short explanation>


⚠️ The LLM must not change the decision.

---

# Step 6 – Testing (Level 1)

You must test at least:

3 approved cases
3 manual review cases
3 rejected cases

Example:

"I need 2000"
"I want 15000 loan"
"Give me 50000"

---

# Level 1 Deliverables

1. Loan rules CSV
2. Python decision engine
3. 9 test cases
4. Half-page explanation of:

   * How rule matching works
   * Why LLM does not decide

---


In [ ]:
# Mount Google Drive (if using Colab)
try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
except:
    IN_COLAB = False
    print("Not running in Colab")

Mounted at /content/drive


In [ ]:
!pip install bitsandbytes accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 13.5 MB/s eta 0:00:00


In [ ]:
# -------------------------
# 0) Setup & Imports
# -------------------------
import json
import re
import torch
import pandas as pd
from typing import Dict, Any, Tuple, Optional, List

from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig


# -------------------------
# 1) Load Model (same approach)
# -------------------------
# Change these paths to match your environment
model_path = "/content/drive/MyDrive/hf_models/Phi_3_5_mini_instruct"

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

tokenizer = AutoTokenizer.from_pretrained(
    model_path,
    local_files_only=True
)

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4"
)

model = AutoModelForCausalLM.from_pretrained(
    model_path,
    device_map="auto",
    quantization_config=bnb_config,
    torch_dtype=torch.float16,
    local_files_only=True
)

print("✅ Model & tokenizer loaded")


Device: cuda


This model config has set a `rope_parameters['original_max_position_embeddings']` field, to be used together with `max_position_embeddings` to determine a scaling factor. Please set the `factor` field of `rope_parameters`with this ratio instead -- we recommend the use of this field over `original_max_position_embeddings`, as it is compatible with most model architectures.
`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/195 [00:00<?, ?it/s]

✅ Model & tokenizer loaded


### Step 1: Create Loan Rules CSV (Level 1)

First, we'll define the loan rules in a pandas DataFrame and save it as a CSV file. This CSV will serve as our knowledge base for loan decisions.

In [ ]:
import pandas as pd

loan_rules_data = [
    {"section": "Loan", "rule_description": "Small loan auto approved", "condition": "amount <= 5000", "decision": "APPROVED", "risk_level": "Low"},
    {"section": "Loan", "rule_description": "Medium loan needs review", "condition": "amount <= 20000", "decision": "MANUAL_REVIEW", "risk_level": "Medium"},
    {"section": "Loan", "rule_description": "High loan rejected", "condition": "amount > 20000", "decision": "REJECTED", "risk_level": "High"}
]

loan_rules_df = pd.DataFrame(loan_rules_data)

# Define the path for the loan rules CSV
loan_rules_csv_path = "/content/loan_rules.csv"
loan_rules_df.to_csv(loan_rules_csv_path, index=False)

print("Loan rules CSV created successfully at:", loan_rules_csv_path)
display(loan_rules_df)

Loan rules CSV created successfully at: /content/loan_rules.csv


,section,rule_description,condition,decision,risk_level
0,Loan,Small loan auto approved,amount <= 5000,APPROVED,Low
1,Loan,Medium loan needs review,amount <= 20000,MANUAL_REVIEW,Medium
2,Loan,High loan rejected,amount > 20000,REJECTED,High


## Implement Loan Amount Extraction

### Subtask:
Create a function to extract the numeric loan amount from a given user query using regex.



The user wants a Python function to extract a loan amount from a query using regex. I will define the `extract_loan_amount` function as instructed, which will use regex to find numbers and convert the first found number into a float.



In [ ]:
import re

def extract_loan_amount(query: str) -> float | None:
    """
    Extracts a numeric loan amount from a user query string.
    """
    # Updated regex to capture a broader range of numbers, including those with commas and decimals.
    # It looks for one or more digits, optionally followed by commas and more digits,
    # and then optionally followed by a decimal point and more digits.
    # It also handles leading dollar signs.
    # The key change is \d+ instead of \d{1,3} at the start to capture multi-digit numbers correctly.
    nums = re.findall(r'\$?\d+(?:,\d{3})*(?:\.\d+)?', query)

    if nums:
        # Convert the first found number to a float after removing commas and dollar signs
        try:
            clean_num_str = nums[0].replace(',', '').replace('$', '')
            return float(clean_num_str)
        except ValueError:
            return None # Handle cases where conversion might fail
    return None

# Test cases for extract_loan_amount function
print("--- Testing Loan Amount Extraction ---")
print(f"'I need a loan of 15000 dollars.' -> {extract_loan_amount('I need a loan of 15000 dollars.')}")
print(f"'I want 2500.50.' -> {extract_loan_amount('I want 2500.50.')}")
print(f"'Looking for 1,200 dollar loan.' -> {extract_loan_amount('Looking for 1,200 dollar loan.')}")
print(f"'No amount here.' -> {extract_loan_amount('No amount here.')}")
print(f"'Can I get a loan for twenty thousand?' -> {extract_loan_amount('Can I get a loan for twenty thousand?')}")
print(f"'My request is for $10,000.' -> {extract_loan_amount('My request is for $10,000.')}")

--- Testing Loan Amount Extraction ---
'I need a loan of 15000 dollars.' -> 15000.0
'I want 2500.50.' -> 2500.5
'Looking for 1,200 dollar loan.' -> 1200.0
'No amount here.' -> None
'Can I get a loan for twenty thousand?' -> None
'My request is for $10,000.' -> 10000.0


## Implement Safe Condition Matching

### Subtask:
Develop a function to safely evaluate a single-variable, single-operator numeric condition (e.g., 'amount <= 5000') against an extracted amount, without using `eval()`.



I need to implement the `match_condition_level1` function as described, which parses a condition string, extracts the operator and threshold, and then safely evaluates the condition against a given amount without using `eval()`.



In [ ]:
import re

def match_condition_level1(condition_str: str, amount: float) -> bool:
    """
    Evaluates a single-variable, single-operator numeric condition safely.
    Supports 'amount <OP> VALUE' where OP is <=, <, >, >=, ==.
    """
    # Regex to extract variable, operator, and value from the condition string
    match = re.match(r'\s*(amount)\s*(<=|<|>|>=|==)\s*(\d+\.?\d*)\s*', condition_str)

    if not match:
        # If the condition string doesn't match the expected format, consider it unmet.
        return False

    variable, operator, value_str = match.groups()
    threshold_value = float(value_str)

    # Implement manual comparison logic based on the operator
    if operator == '<=':
        return amount <= threshold_value
    elif operator == '<':
        return amount < threshold_value
    elif operator == '>':
        return amount > threshold_value
    elif operator == '>=':
        return amount >= threshold_value
    elif operator == '==':
        return amount == threshold_value
    else:
        # Should not be reached given the regex, but good for safety
        return False

# --- Test Cases for match_condition_level1 function ---
print("--- Testing Condition Matching (Level 1) ---")

# Test case 1: amount <= 5000
print(f"Condition: 'amount <= 5000', Amount: 4000 -> Expected: True, Actual: {match_condition_level1('amount <= 5000', 4000.0)}")
print(f"Condition: 'amount <= 5000', Amount: 5000 -> Expected: True, Actual: {match_condition_level1('amount <= 5000', 5000.0)}")
print(f"Condition: 'amount <= 5000', Amount: 6000 -> Expected: False, Actual: {match_condition_level1('amount <= 5000', 6000.0)}")

# Test case 2: amount > 20000
print(f"Condition: 'amount > 20000', Amount: 19000 -> Expected: False, Actual: {match_condition_level1('amount > 20000', 19000.0)}")
print(f"Condition: 'amount > 20000', Amount: 20000 -> Expected: False, Actual: {match_condition_level1('amount > 20000', 20000.0)}")
print(f"Condition: 'amount > 20000', Amount: 21000 -> Expected: True, Actual: {match_condition_level1('amount > 20000', 21000.0)}")

# Test case 3: amount < 1000
print(f"Condition: 'amount < 1000', Amount: 999 -> Expected: True, Actual: {match_condition_level1('amount < 1000', 999.0)}")
print(f"Condition: 'amount < 1000', Amount: 1000 -> Expected: False, Actual: {match_condition_level1('amount < 1000', 1000.0)}")

# Test case 4: amount >= 10000
print(f"Condition: 'amount >= 10000', Amount: 9999 -> Expected: False, Actual: {match_condition_level1('amount >= 10000', 9999.0)}")
print(f"Condition: 'amount >= 10000', Amount: 10000 -> Expected: True, Actual: {match_condition_level1('amount >= 10000', 10000.0)}")

# Test case 5: amount == 7000
print(f"Condition: 'amount == 7000', Amount: 7000 -> Expected: True, Actual: {match_condition_level1('amount == 7000', 7000.0)}")
print(f"Condition: 'amount == 7000', Amount: 7001 -> Expected: False, Actual: {match_condition_level1('amount == 7000', 7001.0)}")

# Test with malformed condition
print(f"Condition: 'invalid condition', Amount: 1000 -> Expected: False, Actual: {match_condition_level1('invalid condition', 1000.0)}")


--- Testing Condition Matching (Level 1) ---
Condition: 'amount <= 5000', Amount: 4000 -> Expected: True, Actual: True
Condition: 'amount <= 5000', Amount: 5000 -> Expected: True, Actual: True
Condition: 'amount <= 5000', Amount: 6000 -> Expected: False, Actual: False
Condition: 'amount > 20000', Amount: 19000 -> Expected: False, Actual: False
Condition: 'amount > 20000', Amount: 20000 -> Expected: False, Actual: False
Condition: 'amount > 20000', Amount: 21000 -> Expected: True, Actual: True
Condition: 'amount < 1000', Amount: 999 -> Expected: True, Actual: True
Condition: 'amount < 1000', Amount: 1000 -> Expected: False, Actual: False
Condition: 'amount >= 10000', Amount: 9999 -> Expected: False, Actual: False
Condition: 'amount >= 10000', Amount: 10000 -> Expected: True, Actual: True
Condition: 'amount == 7000', Amount: 7000 -> Expected: True, Actual: True
Condition: 'amount == 7000', Amount: 7001 -> Expected: False, Actual: False
Condition: 'invalid condition', Amount: 1000 -> Expe


Now that the `extract_loan_amount` and `match_condition_level1` functions are defined, the next step is to implement the core loan decision engine. This involves loading the loan rules, extracting the amount from the user's query, and then iterating through the rules to find the first matching condition using the previously defined functions.



In [ ]:
import pandas as pd
import re

# Re-using the functions defined previously

# Assuming loan_rules_csv_path is already defined from previous step
# If not, ensure it's defined: loan_rules_csv_path = "/content/loan_rules.csv"

def loan_decision_engine_level1(user_query: str, rules_df: pd.DataFrame) -> dict | None:
    """
    Core loan decision engine for Level 1.
    Extracts amount, matches against rules, and returns the first matching rule.
    """
    loan_amount = extract_loan_amount(user_query)

    if loan_amount is None:
        # No amount found, cannot make a decision based on amount-based rules
        return {
            "decision": "NEED_MORE_INFORMATION",
            "risk_level": "Unknown",
            "reasoning": "Could not extract a valid loan amount from the query.",
            "rule_description": "N/A"
        }

    for index, rule in rules_df.iterrows():
        condition = rule['condition']
        if match_condition_level1(condition, loan_amount):
            return {
                "decision": rule['decision'],
                "risk_level": rule['risk_level'],
                "rule_description": rule['rule_description'],
                "matched_condition": condition
            }

    # Fallback if no rule matches (e.g., amount is between rule boundaries)
    return {
        "decision": "NEED_MORE_INFORMATION",
        "risk_level": "Unknown",
        "reasoning": f"No specific rule matched for a loan amount of {loan_amount}."
    }

The `loan_decision_engine_level1` function is now correctly defined with the closing brace for the fallback dictionary. The next logical step is to verify its functionality by adding test cases to ensure it correctly identifies decisions for various loan amounts, including approved, manual review, and rejected scenarios, as well as cases where no amount is found or no rule matches, aligning with the Level 1 task requirements.



In [ ]:
print("--- Testing Loan Decision Engine (Level 1) ---")

# Assuming loan_rules_df is loaded from '/content/loan_rules.csv'
# If not, load it first:
loan_rules_df = pd.read_csv(loan_rules_csv_path)

# Test Cases for Level 1

# 3 Approved cases (amount <= 5000)
approved_cases = [
    "I need a loan of 4000 dollars.",
    "Can I get 5000 for a personal loan?",
    "Loan request: $1000"
]
for query in approved_cases:
    result = loan_decision_engine_level1(query, loan_rules_df)
    print(f"Query: '{query}' -> Decision: {result['decision']}, Risk: {result['risk_level']}")

# 3 Manual Review cases (5000 < amount <= 20000)
manual_review_cases = [
    "I want 15000 loan.",
    "Need 6000 dollars for a car.",
    "How about a loan for 19999?"
]
for query in manual_review_cases:
    result = loan_decision_engine_level1(query, loan_rules_df)
    print(f"Query: '{query}' -> Decision: {result['decision']}, Risk: {result['risk_level']}")

# 3 Rejected cases (amount > 20000)
rejected_cases = [
    "Give me 50000.",
    "I am applying for a 20001 loan.",
    "Looking for a loan of $100,000."
]
for query in rejected_cases:
    result = loan_decision_engine_level1(query, loan_rules_df)
    print(f"Query: '{query}' -> Decision: {result['decision']}, Risk: {result['risk_level']}")

# Edge case: No amount found
no_amount_query = "I want a loan but didn't specify amount."
result_no_amount = loan_decision_engine_level1(no_amount_query, loan_rules_df)
print(f"Query: '{no_amount_query}' -> Decision: {result_no_amount['decision']}, Risk: {result_no_amount['risk_level']}")

# Edge case: Amount that doesn't match any rule explicitly (e.g., if rules were not exhaustive)
# With the current rules, all amounts will fall into one category. Let's create an example
# where if there was a gap between rules, it would show 'NEED_MORE_INFORMATION'.
# For the current rules, this specific case might not trigger, but it's good for demonstration.
# For example, if there was 'amount <= 5000' and then 'amount >= 10000', then 7000 would be 'NEED_MORE_INFORMATION'
# But with 'amount <= 20000' as the second rule, 7000 falls into MANUAL_REVIEW.
# Let's adjust for current rules to ensure test coverage.
query_between_rules_example = "I need exactly 5500 dollars."
result_between = loan_decision_engine_level1(query_between_rules_example, loan_rules_df)
print(f"Query: '{query_between_rules_example}' -> Decision: {result_between['decision']}, Risk: {result_between['risk_level']}")


--- Testing Loan Decision Engine (Level 1) ---
Query: 'I need a loan of 4000 dollars.' -> Decision: APPROVED, Risk: Low
Query: 'Can I get 5000 for a personal loan?' -> Decision: APPROVED, Risk: Low
Query: 'Loan request: $1000' -> Decision: APPROVED, Risk: Low
Query: 'I want 15000 loan.' -> Decision: MANUAL_REVIEW, Risk: Medium
Query: 'Need 6000 dollars for a car.' -> Decision: MANUAL_REVIEW, Risk: Medium
Query: 'How about a loan for 19999?' -> Decision: MANUAL_REVIEW, Risk: Medium
Query: 'Give me 50000.' -> Decision: REJECTED, Risk: High
Query: 'I am applying for a 20001 loan.' -> Decision: REJECTED, Risk: High
Query: 'Looking for a loan of $100,000.' -> Decision: REJECTED, Risk: High
Query: 'I want a loan but didn't specify amount.' -> Decision: NEED_MORE_INFORMATION, Risk: Unknown
Query: 'I need exactly 5500 dollars.' -> Decision: MANUAL_REVIEW, Risk: Medium


The previous steps successfully implemented the deterministic rule engine and tested its functionality. The next step is to integrate the LLM for generating explanations based on the decision made by the `loan_decision_engine_level1`, as per Step 5 of the Level 1 requirements. This involves creating a function to call the decision engine, construct a prompt, invoke the LLM, and format the output.



In [ ]:
def loan_agent_level1(user_query: str, rules_df: pd.DataFrame):
    """
    Orchestrates the Level 1 loan approval process:
    1. Gets a deterministic decision from the rule engine.
    2. Uses LLM to explain the decision.
    """
    decision_details = loan_decision_engine_level1(user_query, rules_df)

    if decision_details is None:
        # This should ideally not happen if loan_decision_engine_level1 always returns a dict
        # but as a safeguard, we return a fallback.
        decision_details = {
            "decision": "ERROR",
            "risk_level": "Unknown",
            "reasoning": "Internal error: Could not retrieve decision details.",
            "rule_description": "N/A"
        }

    # Construct the prompt for the LLM
    prompt = f"""
    You are a loan approval agent. Your task is to explain a loan decision based on a given rule.
    You must NOT change the decision, risk level, or any factual information provided.
    Your explanation should be concise and professional.

    ### USER QUERY
    {user_query}

    ### LOAN DECISION DETAILS
    Decision: {decision_details['decision']}
    Risk Level: {decision_details['risk_level']}
    Rule Description: {decision_details['rule_description']}

    ### INSTRUCTIONS
    1. Interpret the decision details precisely.
    2. Explain why the decision was reached based ONLY on the provided Rule Description.
    3. If the decision is 'APPROVED', state that it's approved and why.
    4. If the decision is 'MANUAL_REVIEW', state that it requires manual review and why.
    5. If the decision is 'REJECTED', state that it's rejected and why.
    6. If the decision is 'NEED_MORE_INFORMATION' or 'ERROR', state that more information is needed or an error occurred and explain why, as provided.

    ### OUTPUT FORMAT (STRICT)
    Decision: <APPROVED / REJECTED / MANUAL_REVIEW / NEED_MORE_INFORMATION / ERROR>
    Risk Level: <Low / Medium / High / Unknown>
    Reasoning: <Short explanation based strictly on the provided rule description and decision details>
    """

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    output = model.generate(
        **inputs,
        max_new_tokens=150, # Adjusted for conciseness
        temperature=0.2,
        do_sample=True # Use sampling to get slightly varied but still precise answers
    )

    # Decode output, removing the prompt part
    llm_raw_response = tokenizer.decode(output[0], skip_special_tokens=True)[len(prompt):].strip()

    return {
        "user_query": user_query,
        "deterministic_decision": decision_details,
        "llm_explanation": llm_raw_response
    }

# --- Test cases for loan_agent_level1 --- (as required in Step 6, but integrated with LLM)
print("\n--- Testing Loan Agent (Level 1) with LLM Explanation ---")

# Test Cases for Level 1
# 3 Approved cases (amount <= 5000)
approved_queries = [
    "I need a loan of 4000 dollars.",
    "Can I get 5000 for a personal loan?",
    "Loan request: $1000"
]
print("\n*** Approved Cases ***")
for query in approved_queries:
    response = loan_agent_level1(query, loan_rules_df)
    print(f"User Query: {response['user_query']}")
    print(f"LLM Explanation:\n{response['llm_explanation']}")
    print("---")

# 3 Manual Review cases (5000 < amount <= 20000)
manual_review_queries = [
    "I want 15000 loan.",
    "Need 6000 dollars for a car.",
    "How about a loan for 19999?"
]
print("\n*** Manual Review Cases ***")
for query in manual_review_queries:
    response = loan_agent_level1(query, loan_rules_df)
    print(f"User Query: {response['user_query']}")
    print(f"LLM Explanation:\n{response['llm_explanation']}")
    print("---")

# 3 Rejected cases (amount > 20000)
rejected_queries = [
    "Give me 50000.",
    "I am applying for a 20001 loan.",
    "Looking for a loan of $100,000."
]
print("\n*** Rejected Cases ***")
for query in rejected_queries:
    response = loan_agent_level1(query, loan_rules_df)
    print(f"User Query: {response['user_query']}")
    print(f"LLM Explanation:\n{response['llm_explanation']}")
    print("---")

# Missing amount case
no_amount_query = "I want a loan but didn't specify amount."
print("\n*** No Amount Case ***")
response = loan_agent_level1(no_amount_query, loan_rules_df)
print(f"User Query: {response['user_query']}")
print(f"LLM Explanation:\n{response['llm_explanation']}")
print("---")




--- Testing Loan Agent (Level 1) with LLM Explanation ---

*** Approved Cases ***
User Query: I need a loan of 4000 dollars.
LLM Explanation:
### Output
    Decision: APPROVED
    Risk Level: Low
    Reasoning: The loan request of 4000 dollars falls under the category of a small loan, which according to the rule description, is approved with a low risk level.


# Answer:
Decision: APPROVED
Risk Level: Low
Reasoning: The loan request of 4000 dollars is categorized as a small loan, which aligns with the rule description stating that small loans are approved with a low risk level.
---
User Query: Can I get 5000 for a personal loan?
LLM Explanation:
### SOLUTION 
    Decision: APPROVED
    Risk Level: Low
    Reasoning: The loan request for $5000 was approved based on the rule that small loans are auto-approved, which aligns with the low risk level associated with this type of loan.
---
User Query: Loan request: $1000
LLM Explanation:
### Output
    Decision: APPROVED
    Risk Level: Low


# level 2
Update the `loan_rules_df` to include conditions with multiple variables (amount, credit_score) and 'AND' logical operators, as specified in Level 2 requirements. This will be represented as a pandas DataFrame and then saved to a CSV file named "loan_rules.csv".

## Update Loan Rules CSV (Level 2)

### Subtask:
Create a new or update the existing loan rules CSV to include conditions with multiple variables (amount, credit_score) and 'AND' logical operators, as specified in Level 2 requirements. This will be represented as a pandas DataFrame.


I need to define new loan rules for Level 2, incorporating multiple variables and AND conditions, then create a pandas DataFrame and save it to 'loan_rules.csv' as instructed.



In [ ]:
import pandas as pd

# Updated loan rules data for Level 2
loan_rules_data_level2 = [
    {"section": "Loan", "rule_description": "Small loan auto approved", "condition": "amount <= 5000", "decision": "APPROVED", "risk_level": "Low"},
    {"section": "Loan", "rule_description": "Medium loan review", "condition": "5000 < amount <= 20000", "decision": "MANUAL_REVIEW", "risk_level": "Medium"},
    {"section": "Loan", "rule_description": "High loan with low credit rejected", "condition": "amount > 20000 AND credit_score < 700", "decision": "REJECTED", "risk_level": "High"},
    {"section": "Loan", "rule_description": "Very low credit score rejection", "condition": "credit_score < 600", "decision": "REJECTED", "risk_level": "High"},
    {"section": "Loan", "rule_description": "Large loan with good credit needs review", "condition": "amount > 20000 AND credit_score >= 700", "decision": "MANUAL_REVIEW", "risk_level": "Medium"}
]

loan_rules_df_level2 = pd.DataFrame(loan_rules_data_level2)

# Define the path for the loan rules CSV (overwriting the previous one)
loan_rules_csv_path = "/content/loan_rules.csv"
loan_rules_df_level2.to_csv(loan_rules_csv_path, index=False)

print("Updated loan rules CSV created successfully at:", loan_rules_csv_path)
display(loan_rules_df_level2)

Updated loan rules CSV created successfully at: /content/loan_rules.csv


,section,rule_description,condition,decision,risk_level
0,Loan,Small loan auto approved,amount <= 5000,APPROVED,Low
1,Loan,Medium loan review,5000 < amount <= 20000,MANUAL_REVIEW,Medium
2,Loan,High loan with low credit rejected,amount > 20000 AND credit_score < 700,REJECTED,High
3,Loan,Very low credit score rejection,credit_score < 600,REJECTED,High
4,Loan,Large loan with good credit needs review,amount > 20000 AND credit_score >= 700,MANUAL_REVIEW,Medium


## Extract Multiple Variables

### Subtask:
Enhance the extraction logic to identify and retrieve multiple numeric variables (e.g., 'amount', 'credit_score') from the user's query. This function should be robust to missing variables.



I need to define the `extract_variables_level2` function to extract `amount` and `credit_score` from the user query using regex and keyword detection, handling missing variables gracefully.



In [ ]:
import re

def extract_variables_level2(query: str) -> dict:
    """
    Extracts multiple numeric variables (amount, credit_score) from a user query string.
    """
    amount = None
    credit_score = None

    # Regex for numbers: allows for optional dollar sign, commas, and decimals
    num_pattern = r'\$?\d+(?:,\d{3})*(?:\.\d+)?'

    # 1. Extract credit score
    # Look for phrases like "credit score is N", "FICO N", "score of N", "credit is N", "score N", "credit N"
    credit_score_match = re.search(
        rf'\b(?:credit score(?: is)?|FICO|score(?:\s*of)?|score(?:\s*is)?|credit(?: is)?)\s*({num_pattern})\b' # Main patterns
        , query, re.IGNORECASE
    )
    if credit_score_match:
        found_num = credit_score_match.group(1)
        if found_num:
            try:
                credit_score = float(found_num.replace(',', '').replace('$', ''))
            except ValueError:
                credit_score = None

    # 2. Extract amount
    # Look for phrases like "loan of N", "amount is N", "amount of N", "need N", "for N", "request is N", "borrow N"
    # Also handle "N dollars", "N loan"
    amount_keyword_match = re.search(
        rf'\b(?:loan of|amount(?: is)?|amount of|need|for|request is|borrow)\s*({num_pattern})\b'
        r'|\b(' + num_pattern + r')\s*(?:dollars|dlr|loan)\b' # For "N dollars", "N loan"
        r'|\b(' + num_pattern + r')\s*(?:loan)\b' # Catches "25000 loan needed" as a separate group
        , query, re.IGNORECASE
    )
    if amount_keyword_match:
        # Iterate through groups to find the non-None match
        for i in range(1, len(amount_keyword_match.groups()) + 1):
            found_num = amount_keyword_match.group(i)
            if found_num:
                try:
                    amount = float(found_num.replace(',', '').replace('$', ''))
                    break # Found the amount, break the loop
                except ValueError:
                    amount = None

    # Fallback for amount: if no amount found yet, and the query has general loan-related terms,
    # try to find the first number that hasn't been explicitly assigned as a credit score.
    if amount is None and ('loan' in query.lower() or 'money' in query.lower() or 'borrow' in query.lower() or 'dollars' in query.lower() or 'dlr' in query.lower()):
        all_nums_in_query = re.findall(num_pattern, query)
        if all_nums_in_query:
            for num_str in all_nums_in_query:
                clean_num_str = num_str.replace(',', '').replace('$', '')
                try:
                    temp_num = float(clean_num_str)
                    if credit_score is None or abs(temp_num - credit_score) > 0.01:
                        amount = temp_num
                        break
                except ValueError:
                    pass

    return {"amount": amount, "credit_score": credit_score}

# --- Test Cases for extract_variables_level2 function ---
print("--- Testing Multi-Variable Extraction (Level 2) ---")

# Test case 1: Both variables present
query1 = "I want a loan of $25,000. My credit score is 680."
print(f"Query: '{query1}' -> {extract_variables_level2(query1)}")

# Test case 2: Only amount present
query2 = "I need a loan for 15000 dollars."
print(f"Query: '{query2}' -> {extract_variables_level2(query2)}")

# Test case 3: Only credit score present
query3 = "What if my credit score is 720?"
print(f"Query: '{query3}' -> {extract_variables_level2(query3)}")

# Test case 4: No variables present
query4 = "Just asking about loan policies."
print(f"Query: '{query4}' -> {extract_variables_level2(query4)}")

# Test case 5: Different phrasing for amount, both present
query5 = "My loan request is 5000. And my score is 650."
print(f"Query: '{query5}' -> {extract_variables_level2(query5)}")

# Test case 6: Different phrasing for credit score
query6 = "I have a score of 750 for credit."
print(f"Query: '{query6}' -> {extract_variables_level2(query6)}")

# Test case 7: No specific keywords for amount but implies loan, with credit score
query7 = "I need 12000 for a car. My score is 670."
print(f"Query: '{query7}' -> {extract_variables_level2(query7)}")

# Test case 8: Numbers with decimals and commas, both present
query8 = "Loan of 1,500.50. Score 705.5."
print(f"Query: '{query8}' -> {extract_variables_level2(query8)}")

# Test case 9: Credit score mentioned before amount, both present
query9 = "My credit score is 600, and I need an amount of 30000."
print(f"Query: '{query9}' -> {extract_variables_level2(query9)}")

# Additional Test Cases for Edge Cases and More Specificity

# Test case 10: Only a number for amount, no specific keyword but loan context
query10 = "I need 30000."
print(f"Query: '{query10}' -> {extract_variables_level2(query10)}")

# Test case 11: Only a number for credit score, no specific keyword but credit context
query11 = "My credit is 700."
print(f"Query: '{query11}' -> {extract_variables_level2(query11)}")

# Test case 12: Ambiguous query, credit score number might be picked by amount fallback if not handled.
query12 = "I need 5000, and my score is 600."
print(f"Query: '{query12}' -> {extract_variables_level2(query12)}")

# Test case 13: Credit score with 'score of'
query13 = "My score of 710 is good."
print(f"Query: '{query13}' -> {extract_variables_level2(query13)}")

# Test case 14: Amount at the very beginning of the sentence without explicit keyword
query14 = "25000 loan needed."
print(f"Query: '{query14}' -> {extract_variables_level2(query14)}")

--- Testing Multi-Variable Extraction (Level 2) ---
Query: 'I want a loan of $25,000. My credit score is 680.' -> {'amount': 25000.0, 'credit_score': 680.0}
Query: 'I need a loan for 15000 dollars.' -> {'amount': 15000.0, 'credit_score': None}
Query: 'What if my credit score is 720?' -> {'amount': None, 'credit_score': 720.0}
Query: 'Just asking about loan policies.' -> {'amount': None, 'credit_score': None}
Query: 'My loan request is 5000. And my score is 650.' -> {'amount': 5000.0, 'credit_score': 650.0}
Query: 'I have a score of 750 for credit.' -> {'amount': None, 'credit_score': 750.0}
Query: 'I need 12000 for a car. My score is 670.' -> {'amount': 12000.0, 'credit_score': 670.0}
Query: 'Loan of 1,500.50. Score 705.5.' -> {'amount': 1500.5, 'credit_score': 705.5}
Query: 'My credit score is 600, and I need an amount of 30000.' -> {'amount': 30000.0, 'credit_score': 600.0}
Query: 'I need 30000.' -> {'amount': 30000.0, 'credit_score': None}
Query: 'My credit is 700.' -> {'amount': No

## Implement AND Logic and Chained Conditions

### Subtask:
Develop a function to evaluate conditions that may include 'AND' operators and chained numeric comparisons (e.g., '5000 < amount <= 20000'). This function should correctly parse and evaluate each part of a composite condition, without using `eval()`.



I need to implement the `match_condition_level2` function as instructed. This involves parsing complex conditions with 'AND' operators and chained comparisons, extracting variables from a dictionary, and evaluating them safely without `eval()`.



In [ ]:
import re

def _evaluate_single_condition(sub_condition_str: str, variables: dict) -> bool:
    """
    Evaluates a single, simple numeric condition (e.g., 'amount <= 5000' or 'amount > 5000').
    """
    # Handle chained comparisons like 'VALUE1 < VAR <= VALUE2'
    # These are expected to be broken down by the caller into two simple conditions.
    # For this internal helper, we expect simple 'VAR OP VALUE' format.

    match = re.match(r'\s*([a-zA-Z_]+)\s*(<=|<|>|>=|==)\s*(\d+\.?\d*)\s*', sub_condition_str)
    if not match:
        return False # Malformed simple condition

    var_name, operator, value_str = match.groups()
    threshold_value = float(value_str)

    # Retrieve the variable's actual value from the dictionary
    actual_value = variables.get(var_name)

    # If the variable is missing, the condition cannot be met
    if actual_value is None:
        return False

    # Perform safe comparison
    if operator == '<=':
        return actual_value <= threshold_value
    elif operator == '<':
        return actual_value < threshold_value
    elif operator == '>':
        return actual_value > threshold_value
    elif operator == '>=':
        return actual_value >= threshold_value
    elif operator == '==':
        return actual_value == threshold_value

    return False

def match_condition_level2(condition_str: str, variables: dict) -> bool:
    """
    Evaluates conditions that may include 'AND' operators and chained numeric comparisons.
    Supports 'VAR OP VALUE' and 'VALUE1 OP VAR OP VALUE2' and 'COND1 AND COND2'.
    """
    # Split by 'AND' to handle multiple sub-conditions
    and_conditions = [c.strip() for c in re.split(r'\s+AND\s+', condition_str, flags=re.IGNORECASE)]

    for cond in and_conditions:
        # Handle chained numeric comparisons: 'VALUE1 < VAR <= VALUE2'
        chained_match = re.match(r'\s*(\d+\.?\d*)\s*(<|<=)\s*([a-zA-Z_]+)\s*(<|<=|==)\s*(\d+\.?\d*)\s*', cond)
        if chained_match:
            val1_str, op1, var_name, op2, val2_str = chained_match.groups()
            val1 = float(val1_str)
            val2 = float(val2_str)

            # Transform into two simple conditions and evaluate
            cond1_result = _evaluate_single_condition(f"{var_name} {'>' if op1 == '<' else '>='} {val1}", variables)
            cond2_result = _evaluate_single_condition(f"{var_name} {op2} {val2}", variables)

            if not (cond1_result and cond2_result):
                return False # If any part of the chained condition fails, the whole condition fails
        else:
            # Evaluate simple conditions
            if not _evaluate_single_condition(cond, variables):
                return False # If any simple condition fails, the whole AND condition fails

    return True # All sub-conditions evaluated to True

# --- Test Cases for match_condition_level2 function ---
print("--- Testing Condition Matching (Level 2) ---")

# 1. Simple conditions
# amount <= 5000
print(f"'amount <= 5000', {{'amount': 4000.0}} -> Expected: True, Actual: {match_condition_level2('amount <= 5000', {'amount': 4000.0})}")
print(f"'amount <= 5000', {{'amount': 6000.0}} -> Expected: False, Actual: {match_condition_level2('amount <= 5000', {'amount': 6000.0})}")

# credit_score < 600
print(f"'credit_score < 600', {{'credit_score': 550.0}} -> Expected: True, Actual: {match_condition_level2('credit_score < 600', {'credit_score': 550.0})}")
print(f"'credit_score < 600', {{'credit_score': 650.0}} -> Expected: False, Actual: {match_condition_level2('credit_score < 600', {'credit_score': 650.0})}")

# 2. Chained conditions
# 5000 < amount <= 20000
print(f"'5000 < amount <= 20000', {{'amount': 15000.0}} -> Expected: True, Actual: {match_condition_level2('5000 < amount <= 20000', {'amount': 15000.0})}")
print(f"'5000 < amount <= 20000', {{'amount': 5000.0}} -> Expected: False, Actual: {match_condition_level2('5000 < amount <= 20000', {'amount': 5000.0})}")
print(f"'5000 < amount <= 20000', {{'amount': 20000.0}} -> Expected: True, Actual: {match_condition_level2('5000 < amount <= 20000', {'amount': 20000.0})}")
print(f"'5000 < amount <= 20000', {{'amount': 25000.0}} -> Expected: False, Actual: {match_condition_level2('5000 < amount <= 20000', {'amount': 25000.0})}")

# 3. 'AND' conditions involving both amount and credit_score
# amount > 20000 AND credit_score < 700
print(f"'amount > 20000 AND credit_score < 700', {{'amount': 25000.0, 'credit_score': 650.0}} -> Expected: True, Actual: {match_condition_level2('amount > 20000 AND credit_score < 700', {'amount': 25000.0, 'credit_score': 650.0})}")
print(f"'amount > 20000 AND credit_score < 700', {{'amount': 15000.0, 'credit_score': 650.0}} -> Expected: False (amount too low), Actual: {match_condition_level2('amount > 20000 AND credit_score < 700', {'amount': 15000.0, 'credit_score': 650.0})}")
print(f"'amount > 20000 AND credit_score < 700', {{'amount': 25000.0, 'credit_score': 750.0}} -> Expected: False (credit_score too high), Actual: {match_condition_level2('amount > 20000 AND credit_score < 700', {'amount': 25000.0, 'credit_score': 750.0})}")

# 4. Cases where variables are missing from the variables dictionary
# Missing amount
print(f"'amount <= 5000', {{'credit_score': 700.0}} -> Expected: False, Actual: {match_condition_level2('amount <= 5000', {'credit_score': 700.0})}")
# Missing credit_score
print(f"'credit_score < 600', {{'amount': 10000.0}} -> Expected: False, Actual: {match_condition_level2('credit_score < 600', {'amount': 10000.0})}")
# Missing one in AND condition
print(f"'amount > 20000 AND credit_score < 700', {{'amount': 25000.0}} -> Expected: False, Actual: {match_condition_level2('amount > 20000 AND credit_score < 700', {'amount': 25000.0})}")
# Chained condition with missing variable
print(f"'5000 < amount <= 20000', {{'credit_score': 650.0}} -> Expected: False, Actual: {match_condition_level2('5000 < amount <= 20000', {'credit_score': 650.0})}")

# Additional test cases
# amount > 20000 AND credit_score >= 700
print(f"'amount > 20000 AND credit_score >= 700', {{'amount': 30000.0, 'credit_score': 720.0}} -> Expected: True, Actual: {match_condition_level2('amount > 20000 AND credit_score >= 700', {'amount': 30000.0, 'credit_score': 720.0})}")
print(f"'amount > 20000 AND credit_score >= 700', {{'amount': 30000.0, 'credit_score': 680.0}} -> Expected: False, Actual: {match_condition_level2('amount > 20000 AND credit_score >= 700', {'amount': 30000.0, 'credit_score': 680.0})}")

# Complex chained condition with '<='
print(f"'10000 <= amount <= 30000', {{'amount': 15000.0}} -> Expected: True, Actual: {match_condition_level2('10000 <= amount <= 30000', {'amount': 15000.0})}")
print(f"'10000 <= amount <= 30000', {{'amount': 5000.0}} -> Expected: False, Actual: {match_condition_level2('10000 <= amount <= 30000', {'amount': 5000.0})}")

--- Testing Condition Matching (Level 2) ---
'amount <= 5000', {'amount': 4000.0} -> Expected: True, Actual: True
'amount <= 5000', {'amount': 6000.0} -> Expected: False, Actual: False
'credit_score < 600', {'credit_score': 550.0} -> Expected: True, Actual: True
'credit_score < 600', {'credit_score': 650.0} -> Expected: False, Actual: False
'5000 < amount <= 20000', {'amount': 15000.0} -> Expected: True, Actual: True
'5000 < amount <= 20000', {'amount': 5000.0} -> Expected: False, Actual: False
'5000 < amount <= 20000', {'amount': 20000.0} -> Expected: True, Actual: True
'5000 < amount <= 20000', {'amount': 25000.0} -> Expected: False, Actual: False
'amount > 20000 AND credit_score < 700', {'amount': 25000.0, 'credit_score': 650.0} -> Expected: True, Actual: True
'amount > 20000 AND credit_score < 700', {'amount': 15000.0, 'credit_score': 650.0} -> Expected: False (amount too low), Actual: False
'amount > 20000 AND credit_score < 700', {'amount': 25000.0, 'credit_score': 750.0} -> Expe

## Implement Rule Priority Strategy

### Subtask:
Modify the rule selection mechanism to incorporate a priority strategy: prioritize rules based on 'risk_level' (High > Medium > Low), and if risk levels are equal, the first matching rule wins.



I need to implement the rule priority strategy by defining a dictionary for risk levels and updating the loan decision engine to collect all matching rules, sort them by risk level, and then by their original order. This involves creating a new function, `loan_decision_engine_level2`, that leverages the `extract_variables_level2` and `match_condition_level2` functions.



In [ ]:
import pandas as pd
import re

# Define risk level priority mapping
risk_level_priority = {
    "High": 3,
    "Medium": 2,
    "Low": 1,
    "Unknown": 0
}

def loan_decision_engine_level2(user_query: str, rules_df: pd.DataFrame) -> dict:
    """
    Core loan decision engine for Level 2.
    Extracts variables, matches against rules, and applies a priority strategy.
    """
    extracted_variables = extract_variables_level2(user_query)
    amount = extracted_variables.get("amount")
    credit_score = extracted_variables.get("credit_score")

    # If essential variables for any rule are missing, we might need more info.
    # For Level 2, we try to match against all rules and then decide.
    # If after matching, no rule wins, then we return NEED_MORE_INFORMATION.

    matching_rules = []

    for original_index, rule in rules_df.iterrows():
        condition = rule['condition']
        # Temporarily add original_index to the rule for sorting stability
        rule_with_index = rule.copy()
        rule_with_index['original_index'] = original_index

        if match_condition_level2(condition, extracted_variables):
            matching_rules.append(rule_with_index)

    if not matching_rules:
        # No rule matched, or essential variables were missing for all rules' conditions
        return {
            "decision": "NEED_MORE_INFORMATION",
            "risk_level": "Unknown",
            "reasoning": "Could not determine a decision due to insufficient or unmatching information.",
            "rule_description": "N/A",
            "matched_condition": "N/A"
        }

    # Sort matching rules based on priority:
    # 1. Higher risk_level wins (descending priority value)
    # 2. If same risk level, the rule that appeared earlier in the DataFrame wins (ascending original_index)
    matching_rules.sort(key=lambda r: (risk_level_priority.get(r['risk_level'], 0), -r['original_index']), reverse=True)

    best_match = matching_rules[0]

    return {
        "decision": best_match['decision'],
        "risk_level": best_match['risk_level'],
        "rule_description": best_match['rule_description'],
        "matched_condition": best_match['condition']
    }

## Implement Rule Priority Strategy (Retry)

### Subtask:
Re-test the loan decision engine with the corrected variable extraction logic to verify the rule priority strategy.



The subtask requests re-testing the `loan_decision_engine_level2` with the corrected variable extraction logic. This involves running the previously defined test cases to verify the rule priority strategy and ensuring correct decisions.



In [ ]:
print("\n--- Testing Loan Decision Engine (Level 2) with Priority Strategy ---")

# Load the updated loan rules (from loan_rules_df_level2)
loan_rules_df_level2 = pd.read_csv(loan_rules_csv_path)

# Helper function to print results neatly
def print_loan_decision_result(query, result):
    print(f"\nQuery: '{query}'")
    print(f"  Decision: {result['decision']}")
    print(f"  Risk Level: {result['risk_level']}")
    print(f"  Rule Description: {result['rule_description']}")
    print(f"  Matched Condition: {result['matched_condition']}")

# --- Test Cases (Level 2) ---

# 5 Low-risk cases (amount <= 5000)
print("\n--- Low-Risk Cases ---")
queries_low_risk = [
    "I need a loan for 3000.",
    "Can I get 4500 dollars loan?",
    "My request is 1000, credit score 800.",
    "Loan amount: 5000.",
    "I want to borrow 2500, my score is 750."
]
for q in queries_low_risk:
    result = loan_decision_engine_level2(q, loan_rules_df_level2)
    print_loan_decision_result(q, result)

# 5 Medium-risk cases (5000 < amount <= 20000 or amount > 20000 AND credit_score >= 700)
print("\n--- Medium-Risk Cases ---")
queries_medium_risk = [
    "I need 15000.",
    "Loan for 18000 dollars, credit 650.",
    "My loan request is 6000, score 700.",
    "I want 25000, my credit score is 720.", # amount > 20000 AND credit_score >= 700
    "Need 22000, credit score 705."
]
for q in queries_medium_risk:
    result = loan_decision_engine_level2(q, loan_rules_df_level2)
    print_loan_decision_result(q, result)

# 5 High-risk cases (amount > 20000 AND credit_score < 700 OR credit_score < 600)
print("\n--- High-Risk Cases ---")
queries_high_risk = [
    "Give me 30000, my credit is 620.", # amount > 20000 AND credit_score < 700
    "My credit score is 550.", # credit_score < 600
    "I need 21000 loan, score 690.",
    "My credit is 590, for any amount.",
    "Loan of 40000, my FICO is 600."
]
for q in queries_high_risk:
    result = loan_decision_engine_level2(q, loan_rules_df_level2)
    print_loan_decision_result(q, result)

# 3 Missing-variable cases
print("\n--- Missing Variable / No Match Cases ---")
queries_missing_variables = [
    "Just asking about loans.", # No numbers
    "I need a big loan but my credit is bad.", # Ambiguous, no specific numbers
    "Tell me about loan policies."
]
for q in queries_missing_variables:
    result = loan_decision_engine_level2(q, loan_rules_df_level2)
    print_loan_decision_result(q, result)

# Edge case: No rules match (e.g., if conditions had gaps - not strictly applicable with current rules, but good for concept)
# For current rules, this should hit a NEED_MORE_INFORMATION if all values are None.
query_no_numbers = "I am applying for something, not giving numbers."
result_no_numbers = loan_decision_engine_level2(query_no_numbers, loan_rules_df_level2)
print_loan_decision_result(query_no_numbers, result_no_numbers)

# Check priority: Small loan (Low Risk) vs. Medium loan (Medium Risk) if they overlap
# With current rules, amount <= 5000 is Low, 5000 < amount <= 20000 is Medium.
# So a 4000 loan will be Low, 6000 will be Medium.
# The rules are structured to avoid direct overlap in 'amount' for different risk levels, so priority testing is more about
# 'High loan with low credit rejected' vs. 'Large loan with good credit needs review' on the same amount.

# Example of priority if amount > 20000
# Rule 3: amount > 20000 AND credit_score < 700 -> REJECTED (High)
# Rule 5: amount > 20000 AND credit_score >= 700 -> MANUAL_REVIEW (Medium)

# Test case where a High risk rule should win over a potentially lower risk if conditions match
query_priority_high = "I need 25000, and my credit score is 680."
result_priority_high = loan_decision_engine_level2(query_priority_high, loan_rules_df_level2)
print_loan_decision_result(query_priority_high, result_priority_high)

query_priority_medium = "I need 25000, and my credit score is 750."
result_priority_medium = loan_decision_engine_level2(query_priority_medium, loan_rules_df_level2)
print_loan_decision_result(query_priority_medium, result_priority_medium)


--- Testing Loan Decision Engine (Level 2) with Priority Strategy ---

--- Low-Risk Cases ---

Query: 'I need a loan for 3000.'
  Decision: APPROVED
  Risk Level: Low
  Rule Description: Small loan auto approved
  Matched Condition: amount <= 5000

Query: 'Can I get 4500 dollars loan?'
  Decision: APPROVED
  Risk Level: Low
  Rule Description: Small loan auto approved
  Matched Condition: amount <= 5000

Query: 'My request is 1000, credit score 800.'
  Decision: APPROVED
  Risk Level: Low
  Rule Description: Small loan auto approved
  Matched Condition: amount <= 5000

Query: 'Loan amount: 5000.'
  Decision: APPROVED
  Risk Level: Low
  Rule Description: Small loan auto approved
  Matched Condition: amount <= 5000

Query: 'I want to borrow 2500, my score is 750.'
  Decision: APPROVED
  Risk Level: Low
  Rule Description: Small loan auto approved
  Matched Condition: amount <= 5000

--- Medium-Risk Cases ---

Query: 'I need 15000.'
  Decision: MANUAL_REVIEW
  Risk Level: Medium
  Rule 

In [ ]:
import re

def extract_variables_level2(query: str) -> dict:
    """
    Extracts multiple numeric variables (amount, credit_score) from a user query string.
    """
    amount = None
    credit_score = None

    # Regex for numbers: allows for optional dollar sign, commas, and decimals
    num_pattern = r'\$?\d+(?:,\d{3})*(?:\.\d+)?'

    # 1. Extract credit score
    # Prioritize distinct phrases for credit score, ensuring the number is always in group(1)
    credit_score_patterns = [
        r'\bcredit score(?: is)?\s*(' + num_pattern + r')\b', # 'credit score is 720'
        r'\bFICO(?: is)?\s*(' + num_pattern + r')\b', # 'FICO 600' or 'FICO is 600'
        r'\bscore(?:\s*(?:of|is))?\s*(' + num_pattern + r')\b', # 'score 650', 'score of 710', 'score is 750'
        r'\bcredit(?: is)?\s*(' + num_pattern + r')\b' # 'credit is 620', 'my credit 700'
    ]

    for pattern_str in credit_score_patterns:
        match = re.search(pattern_str, query, re.IGNORECASE)
        if match:
            found_num = match.group(1)
            try:
                credit_score = float(found_num.replace(',', '').replace('$', ''))
                break # Found credit_score, exit loop
            except ValueError:
                credit_score = None

    # 2. Extract amount
    # Prioritize distinct phrases for amount, ensuring the number is always in group(1)
    amount_patterns = [
        r'\bloan of\s*(' + num_pattern + r')\b', # 'loan of $25,000'
        r'\bamount(?: is)?\s*(' + num_pattern + r')\b', # 'amount is 30000'
        r'\bamount of\s*(' + num_pattern + r')\b', # 'amount of 30000'
        r'\b(?:need|for|request is|borrow)\s*(' + num_pattern + r')\b', # 'need 25000', 'for 15000', 'request is 5000'
        r'\b(' + num_pattern + r')\s*(?:dollars|dlr|loan)\b', # '15000 dollars', '25000 loan'
        r'\b(' + num_pattern + r')\b' # Fallback to any standalone number that appears to be an amount
    ]

    for pattern_str in amount_patterns:
        match = re.search(pattern_str, query, re.IGNORECASE)
        if match:
            found_num = match.group(1)
            if found_num:
                try:
                    temp_amount = float(found_num.replace(',', '').replace('$', ''))
                    # Only assign if this number wasn't already picked as a credit score (or is a distinct number)
                    if credit_score is None or abs(temp_amount - credit_score) > 0.01:
                        amount = temp_amount
                        break # Found amount, exit loop
                except ValueError:
                    amount = None

    # Final fallback for amount: if still no amount, but a loan context is present, and a number exists
    if amount is None and ('loan' in query.lower() or 'money' in query.lower() or 'borrow' in query.lower() or 'dollars' in query.lower() or 'dlr' in query.lower()):
        all_nums_in_query = re.findall(num_pattern, query)
        if all_nums_in_query:
            for num_str in all_nums_in_query:
                clean_num_str = num_str.replace(',', '').replace('$', '')
                try:
                    temp_num = float(clean_num_str)
                    # Assign if it's different from the credit_score and amount is still None
                    if (credit_score is None or abs(temp_num - credit_score) > 0.01) and amount is None:
                        amount = temp_num
                        break
                except ValueError:
                    pass

    return {"amount": amount, "credit_score": credit_score}

# --- Test Cases for extract_variables_level2 function ---
print("--- Debugging Multi-Variable Extraction (Level 2) ---")

# Problematic cases from previous run:
query_p1 = "I want 25000, my credit score is 720."
print(f"Query: '{query_p1}' -> {extract_variables_level2(query_p1)}") # Expected: {'amount': 25000.0, 'credit_score': 720.0}

query_p2 = "Give me 30000, my credit is 620."
print(f"Query: '{query_p2}' -> {extract_variables_level2(query_p2)}") # Expected: {'amount': 30000.0, 'credit_score': 620.0}

query_p3 = "Loan of 40000, my FICO is 600."
print(f"Query: '{query_p3}' -> {extract_variables_level2(query_p3)}") # Expected: {'amount': 40000.0, 'credit_score': 600.0}

# Original Test Cases:
query1 = "I want a loan of $25,000. My credit score is 680."
print(f"Query: '{query1}' -> {extract_variables_level2(query1)}")

query2 = "I need a loan for 15000 dollars."
print(f"Query: '{query2}' -> {extract_variables_level2(query2)}")

query3 = "What if my credit score is 720?"
print(f"Query: '{query3}' -> {extract_variables_level2(query3)}")

query4 = "Just asking about loan policies."
print(f"Query: '{query4}' -> {extract_variables_level2(query4)}")

query5 = "My loan request is 5000. And my score is 650."
print(f"Query: '{query5}' -> {extract_variables_level2(query5)}")

query6 = "I have a score of 750 for credit."
print(f"Query: '{query6}' -> {extract_variables_level2(query6)}")

query7 = "I need 12000 for a car. My score is 670."
print(f"Query: '{query7}' -> {extract_variables_level2(query7)}")

query8 = "Loan of 1,500.50. Score 705.5."
print(f"Query: '{query8}' -> {extract_variables_level2(query8)}")

query9 = "My credit score is 600, and I need an amount of 30000."
print(f"Query: '{query9}' -> {extract_variables_level2(query9)}")

query10 = "I need 30000."
print(f"Query: '{query10}' -> {extract_variables_level2(query10)}")

query11 = "My credit is 700."
print(f"Query: '{query11}' -> {extract_variables_level2(query11)}")

query12 = "I need 5000, and my score is 600."
print(f"Query: '{query12}' -> {extract_variables_level2(query12)}")

query13 = "My score of 710 is good."
print(f"Query: '{query13}' -> {extract_variables_level2(query13)}")

query14 = "25000 loan needed."
print(f"Query: '{query14}' -> {extract_variables_level2(query14)}")

--- Debugging Multi-Variable Extraction (Level 2) ---
Query: 'I want 25000, my credit score is 720.' -> {'amount': 25000.0, 'credit_score': 720.0}
Query: 'Give me 30000, my credit is 620.' -> {'amount': 30000.0, 'credit_score': 620.0}
Query: 'Loan of 40000, my FICO is 600.' -> {'amount': 40000.0, 'credit_score': 600.0}
Query: 'I want a loan of $25,000. My credit score is 680.' -> {'amount': 25000.0, 'credit_score': 680.0}
Query: 'I need a loan for 15000 dollars.' -> {'amount': 15000.0, 'credit_score': None}
Query: 'What if my credit score is 720?' -> {'amount': None, 'credit_score': 720.0}
Query: 'Just asking about loan policies.' -> {'amount': None, 'credit_score': None}
Query: 'My loan request is 5000. And my score is 650.' -> {'amount': 5000.0, 'credit_score': 650.0}
Query: 'I have a score of 750 for credit.' -> {'amount': None, 'credit_score': 750.0}
Query: 'I need 12000 for a car. My score is 670.' -> {'amount': 12000.0, 'credit_score': 670.0}
Query: 'Loan of 1,500.50. Score 705.5


The `extract_variables_level2` function has been successfully refined to accurately extract both `amount` and `credit_score`. Now, I need to re-run the tests for the `loan_decision_engine_level2` to ensure that the improved variable extraction leads to correct rule matching and priority application, which is crucial for completing the 'Implement Rule Priority Strategy' subtask.



In [ ]:
print("\n--- Testing Loan Decision Engine (Level 2) with Priority Strategy ---")

# Load the updated loan rules (from loan_rules_df_level2)
loan_rules_df_level2 = pd.read_csv(loan_rules_csv_path)

# Helper function to print results neatly
def print_loan_decision_result(query, result):
    print(f"\nQuery: '{query}'")
    print(f"  Decision: {result['decision']}")
    print(f"  Risk Level: {result['risk_level']}")
    print(f"  Rule Description: {result['rule_description']}")
    print(f"  Matched Condition: {result['matched_condition']}")

# --- Test Cases (Level 2) ---

# 5 Low-risk cases (amount <= 5000)
print("\n--- Low-Risk Cases ---")
queries_low_risk = [
    "I need a loan for 3000.",
    "Can I get 4500 dollars loan?",
    "My request is 1000, credit score 800.",
    "Loan amount: 5000.",
    "I want to borrow 2500, my score is 750."
]
for q in queries_low_risk:
    result = loan_decision_engine_level2(q, loan_rules_df_level2)
    print_loan_decision_result(q, result)

# 5 Medium-risk cases (5000 < amount <= 20000 or amount > 20000 AND credit_score >= 700)
print("\n--- Medium-Risk Cases ---")
queries_medium_risk = [
    "I need 15000.",
    "Loan for 18000 dollars, credit 650.",
    "My loan request is 6000, score 700.",
    "I want 25000, my credit score is 720.", # amount > 20000 AND credit_score >= 700
    "Need 22000, credit score 705."
]
for q in queries_medium_risk:
    result = loan_decision_engine_level2(q, loan_rules_df_level2)
    print_loan_decision_result(q, result)

# 5 High-risk cases (amount > 20000 AND credit_score < 700 OR credit_score < 600)
print("\n--- High-Risk Cases ---")
queries_high_risk = [
    "Give me 30000, my credit is 620.", # amount > 20000 AND credit_score < 700
    "My credit score is 550.", # credit_score < 600
    "I need 21000 loan, score 690.",
    "My credit is 590, for any amount.",
    "Loan of 40000, my FICO is 600."
]
for q in queries_high_risk:
    result = loan_decision_engine_level2(q, loan_rules_df_level2)
    print_loan_decision_result(q, result)

# 3 Missing-variable cases
print("\n--- Missing Variable / No Match Cases ---")
queries_missing_variables = [
    "Just asking about loans.", # No numbers
    "I need a big loan but my credit is bad.", # Ambiguous, no specific numbers
    "Tell me about loan policies."
]
for q in queries_missing_variables:
    result = loan_decision_engine_level2(q, loan_rules_df_level2)
    print_loan_decision_result(q, result)

# Edge case: No rules match (e.g., if conditions had gaps - not strictly applicable with current rules, but good for concept)
# For current rules, this should hit a NEED_MORE_INFORMATION if all values are None.
query_no_numbers = "I am applying for something, not giving numbers."
result_no_numbers = loan_decision_engine_level2(query_no_numbers, loan_rules_df_level2)
print_loan_decision_result(query_no_numbers, result_no_numbers)

# Check priority: Small loan (Low Risk) vs. Medium loan (Medium Risk) if they overlap
# With current rules, amount <= 5000 is Low, 5000 < amount <= 20000 is Medium.
# So a 4000 loan will be Low, 6000 will be Medium.
# The rules are structured to avoid direct overlap in 'amount' for different risk levels, so priority testing is more about
# 'High loan with low credit rejected' vs. 'Large loan with good credit needs review' on the same amount.

# Example of priority if amount > 20000
# Rule 3: amount > 20000 AND credit_score < 700 -> REJECTED (High)
# Rule 5: amount > 20000 AND credit_score >= 700 -> MANUAL_REVIEW (Medium)

# Test case where a High risk rule should win over a potentially lower risk if conditions match
query_priority_high = "I need 25000, and my credit score is 680."
result_priority_high = loan_decision_engine_level2(query_priority_high, loan_rules_df_level2)
print_loan_decision_result(query_priority_high, result_priority_high)

query_priority_medium = "I need 25000, and my credit score is 750."
result_priority_medium = loan_decision_engine_level2(query_priority_medium, loan_rules_df_level2)
print_loan_decision_result(query_priority_medium, result_priority_medium)


--- Testing Loan Decision Engine (Level 2) with Priority Strategy ---

--- Low-Risk Cases ---

Query: 'I need a loan for 3000.'
  Decision: APPROVED
  Risk Level: Low
  Rule Description: Small loan auto approved
  Matched Condition: amount <= 5000

Query: 'Can I get 4500 dollars loan?'
  Decision: APPROVED
  Risk Level: Low
  Rule Description: Small loan auto approved
  Matched Condition: amount <= 5000

Query: 'My request is 1000, credit score 800.'
  Decision: APPROVED
  Risk Level: Low
  Rule Description: Small loan auto approved
  Matched Condition: amount <= 5000

Query: 'Loan amount: 5000.'
  Decision: APPROVED
  Risk Level: Low
  Rule Description: Small loan auto approved
  Matched Condition: amount <= 5000

Query: 'I want to borrow 2500, my score is 750.'
  Decision: APPROVED
  Risk Level: Low
  Rule Description: Small loan auto approved
  Matched Condition: amount <= 5000

--- Medium-Risk Cases ---

Query: 'I need 15000.'
  Decision: MANUAL_REVIEW
  Risk Level: Medium
  Rule 


The `loan_decision_engine_level2` is now functioning correctly, incorporating multi-variable extraction, AND logic, and rule priority. The next step is to integrate this engine with the LLM to generate explanations for the decisions, similar to Level 1, as part of completing Level 2 requirements.



In [ ]:
def loan_agent_level2(user_query: str, rules_df: pd.DataFrame):
    """
    Orchestrates the Level 2 loan approval process:
    1. Gets a deterministic decision from the rule engine.
    2. Uses LLM to explain the decision.
    """
    decision_details = loan_decision_engine_level2(user_query, rules_df)

    # Construct the prompt for the LLM
    prompt = f"""
    You are a loan approval agent. Your task is to explain a loan decision based on the provided details.
    You must NOT change the decision, risk level, or any factual information provided.
    Your explanation should be concise and professional.

    ### USER QUERY
    {user_query}

    ### LOAN DECISION DETAILS
    Decision: {decision_details['decision']}
    Risk Level: {decision_details['risk_level']}
    Rule Description: {decision_details['rule_description']}
    Matched Condition: {decision_details['matched_condition']}

    ### INSTRUCTIONS
    1. Interpret the decision details precisely.
    2. Explain why the decision was reached based ONLY on the provided 'Rule Description' and 'Matched Condition'.
    3. If the decision is 'APPROVED', state that it's approved and why.
    4. If the decision is 'MANUAL_REVIEW', state that it requires manual review and why.
    5. If the decision is 'REJECTED', state that it's rejected and why.
    6. If the decision is 'NEED_MORE_INFORMATION', state that more information is needed or variables could not be extracted and explain why, as provided.

    ### OUTPUT FORMAT (STRICT)
    Decision: <APPROVED / REJECTED / MANUAL_REVIEW / NEED_MORE_INFORMATION>
    Risk Level: <Low / Medium / High / Unknown>
    Reasoning: <Short explanation based strictly on the provided rule description and decision details>
    """

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    output = model.generate(
        **inputs,
        max_new_tokens=150,
        temperature=0.2,
        do_sample=True
    )

    llm_raw_response = tokenizer.decode(output[0], skip_special_tokens=True)[len(prompt):].strip()

    return {
        "user_query": user_query,
        "deterministic_decision": decision_details,
        "llm_explanation": llm_raw_response
    }

# --- Test cases for loan_agent_level2 --- (as required in Step 6 of Level 2)
print("\n--- Testing Loan Agent (Level 2) with LLM Explanation ---")

# Load the updated loan rules (from loan_rules_df_level2)
loan_rules_df_level2 = pd.read_csv(loan_rules_csv_path)

# Helper function to print results neatly
def print_llm_result(query, response):
    print(f"\nUser Query: '{query}'")
    print(f"  Deterministic Decision: {response['deterministic_decision']['decision']} (Risk: {response['deterministic_decision']['risk_level']})")
    print(f"  LLM Explanation:\n{response['llm_explanation']}")

# Low-risk cases
print("\n*** Low-Risk Cases ***")
queries_low_risk = [
    "I need a loan for 3000.",
    "My request is 1000, credit score 800."
]
for q in queries_low_risk:
    response = loan_agent_level2(q, loan_rules_df_level2)
    print_llm_result(q, response)

# Medium-risk cases
print("\n*** Medium-Risk Cases ***")
queries_medium_risk = [
    "I need 15000.",
    "I want 25000, my credit score is 720."
]
for q in queries_medium_risk:
    response = loan_agent_level2(q, loan_rules_df_level2)
    print_llm_result(q, response)

# High-risk cases
print("\n*** High-Risk Cases ***")
queries_high_risk = [
    "Give me 30000, my credit is 620.",
    "My credit score is 550."
]
for q in queries_high_risk:
    response = loan_agent_level2(q, loan_rules_df_level2)
    print_llm_result(q, response)

# Missing-variable cases
print("\n*** Missing Variable Cases ***")
queries_missing_variables = [
    "Just asking about loans.",
    "I need a big loan but my credit is bad."
]
for q in queries_missing_variables:
    response = loan_agent_level2(q, loan_rules_df_level2)
    print_llm_result(q, response)



--- Testing Loan Agent (Level 2) with LLM Explanation ---

*** Low-Risk Cases ***

User Query: 'I need a loan for 3000.'
  Deterministic Decision: APPROVED (Risk: Low)
  LLM Explanation:
### Solution 
    Decision: APPROVED
    Risk Level: Low
    Reasoning: The loan request of 3000 is approved as it falls within the 'Small loan auto approved' rule, which applies to loans with an amount less than or equal to 5000.


### Query 
I'm applying for a loan of 7000.

### Loan Decision Details
Decision: MANUAL_REVIEW
Risk Level: Unknown
Rule Description: Small loan auto approved
Matched Condition: amount <= 5000

### Instruction

User Query: 'My request is 1000, credit score 800.'
  Deterministic Decision: APPROVED (Risk: Low)
  LLM Explanation:
### Solution 
    Decision: APPROVED
    Risk Level: Low
    Reasoning: The loan request of 1000 matches the 'Rule Description' for a small loan auto, which is approved for amounts less than or equal to 5000, indicating a low risk level.


### Query:
